In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [2]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 18.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 309kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.64MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.5MB/s]


In [3]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [6]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [7]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [8]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.288993  [   64/60000]
loss: 2.284149  [ 6464/60000]
loss: 2.261119  [12864/60000]
loss: 2.265856  [19264/60000]
loss: 2.250315  [25664/60000]
loss: 2.208283  [32064/60000]
loss: 2.231827  [38464/60000]
loss: 2.190295  [44864/60000]
loss: 2.191522  [51264/60000]
loss: 2.157657  [57664/60000]
Test Error: 
 Accuracy: 41.1%, Avg loss: 2.152691 

Epoch 2
-------------------------------
loss: 2.158405  [   64/60000]
loss: 2.153496  [ 6464/60000]
loss: 2.092231  [12864/60000]
loss: 2.115140  [19264/60000]
loss: 2.076113  [25664/60000]
loss: 1.998560  [32064/60000]
loss: 2.046540  [38464/60000]
loss: 1.960262  [44864/60000]
loss: 1.968052  [51264/60000]
loss: 1.900713  [57664/60000]
Test Error: 
 Accuracy: 59.0%, Avg loss: 1.892991 

Epoch 3
-------------------------------
loss: 1.919683  [   64/60000]
loss: 1.894872  [ 6464/60000]
loss: 1.774459  [12864/60000]
loss: 1.821542  [19264/60000]
loss: 1.727395  [25664/60000]
loss: 1.657441  [32064/600

In [9]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [10]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [11]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


🎓 내 버전: MNIST Digit Classifier
Tutorial은 옷 종류 (FashionMNIST) 분류했는데, 나는 숫자 (0~9) 분류할거야. 구조는 같고 dataset만 바꾼 느낌



In [13]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

# ── 1. DATA 준비 ─────────────────────────────────────────
# MNIST = 손으로 쓴 숫자 이미지 dataset (28x28 흑백)
training_data = datasets.MNIST(
    root="data", train=True, download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)
test_data = datasets.MNIST(
    root="data", train=False, download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

# DataLoader: 데이터를 batch 단위로 쪼개서 먹여주는 역할
# batch_size=64 → 한번에 64장씩 처리
train_loader = DataLoader(training_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,     batch_size=64)


# ── 2. MODEL 정의 ─────────────────────────────────────────
# GPU 있으면 GPU, 없으면 CPU
device = torch.accelerator.current_accelerator().type \
         if torch.accelerator.is_available() else "cpu"
print(f"Using {device}")

class MyDigitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()          # 28x28 이미지 → 784짜리 1D vector로 펼치기
        self.net = nn.Sequential(
            nn.Linear(784, 256),             # 784 input → 256 hidden nodes
            nn.ReLU(),                       # 음수는 0으로 날려버리는 activation
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)               # 최종 output: 0~9 총 10개 class
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.net(x)

model = MyDigitNet().to(device)
print(model)


# ── 3. LOSS & OPTIMIZER ───────────────────────────────────
# CrossEntropyLoss: 분류 문제에서 standard로 쓰는 loss function
# SGD optimizer: gradient 방향으로 파라미터 업데이트
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)


# ── 4. TRAIN & TEST LOOP ──────────────────────────────────
def train(loader, model, loss_fn, optimizer):
    model.train()                            # dropout 같은거 train 모드로 켜기
    for batch, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)
        pred = model(X)                      # forward pass
        loss = loss_fn(pred, y)              # loss 계산
        loss.backward()                      # backprop: gradient 계산
        optimizer.step()                     # weight 업데이트
        optimizer.zero_grad()                # gradient 초기화 (안하면 누적됨!)
        if batch % 200 == 0:
            print(f"  loss: {loss.item():.4f}  [{batch*len(X)}/{len(loader.dataset)}]")

def test(loader, model, loss_fn):
    model.eval()                             # eval 모드: gradient 계산 안 함
    correct, total_loss = 0, 0
    with torch.no_grad():                    # 메모리 절약
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
    acc = 100 * correct / len(loader.dataset)
    print(f"  Test Accuracy: {acc:.1f}%,  Avg Loss: {total_loss/len(loader):.4f}\n")


# ── 5. TRAINING 실행 ──────────────────────────────────────
for epoch in range(5):
    print(f"Epoch {epoch+1} ─────────────")
    train(train_loader, model, loss_fn, optimizer)
    test(test_loader,  model, loss_fn)
print("Done! 🎉")


# ── 6. 저장 & 로드 ────────────────────────────────────────
torch.save(model.state_dict(), "mnist_model.pth")
print("Model saved!")

# 로드할 때
model2 = MyDigitNet().to(device)
model2.load_state_dict(torch.load("mnist_model.pth", weights_only=True))


# ── 7. 실제 prediction ────────────────────────────────────
model2.eval()
x, y = test_data[0]                         # test set 첫 번째 이미지
with torch.no_grad():
    pred = model2(x.to(device))
    predicted = pred.argmax(1).item() # Changed from argmax(0) to argmax(1)
print(f'Predicted: {predicted},  Actual: {y}')

Using cpu
MyDigitNet(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (net): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)
Epoch 1 ─────────────
  loss: 2.2990  [0/60000]
  loss: 2.2945  [12800/60000]
  loss: 2.2892  [25600/60000]
  loss: 2.2749  [38400/60000]
  loss: 2.2738  [51200/60000]
  Test Accuracy: 23.7%,  Avg Loss: 2.2676

Epoch 2 ─────────────
  loss: 2.2743  [0/60000]
  loss: 2.2514  [12800/60000]
  loss: 2.2564  [25600/60000]
  loss: 2.2333  [38400/60000]
  loss: 2.2309  [51200/60000]
  Test Accuracy: 45.8%,  Avg Loss: 2.2189

Epoch 3 ─────────────
  loss: 2.2264  [0/60000]
  loss: 2.2204  [12800/60000]
  loss: 2.2088  [25600/60000]
  loss: 2.1616  [38400/60000]
  loss: 2.1653  [51200/60000]
  Test Accuracy: 58.3%,  Avg Loss: 2.1409

Epoch 4 ─────────────
  loss: 2.1433  [0/60000]
  loss: